## 0. Setup shared by the three validation notebooks
Paths are relative to the repository root. The statistics come from `tools/validation_metrics.py`
(Ministry of Transport guideline, draft 6, September 2024); results go to `Output/validation_mot/`.

In [1]:
import os, sys, warnings
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'
sys.path.insert(0, 'tools')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import validation_metrics as vm
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

BLUE, ORANGE, AQUA, PURPLE = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6'
INK, INK2, MUTED, GRID, AXIS = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
OUT = 'Output/validation_mot'; FIG = 'Output/figures'
# Time window (wave 1b): AM 06:00-09:00 is the base and the default; PM 16:00-19:00 and midday 10:00-14:00 with the environment variable NOFIT_PERIOD
PERIOD = os.environ.get('NOFIT_PERIOD', 'AM')
HOURS, SUF = {'AM': ([6, 7, 8], ''), 'PM': ([16, 17, 18], '_pm'), 'MD': ([10, 11, 12, 13], '_md')}[PERIOD]
PLABEL = {'AM': 'AM 06:00-09:00', 'PM': 'PM 16:00-19:00', 'MD': 'midday 10:00-14:00'}[PERIOD]
W = {'AM': '06-09', 'PM': '16-19', 'MD': '10-14'}[PERIOD]
PSUF = '' if PERIOD == 'AM' else '_' + PERIOD            # suffix of the validation output files of the other periods
def vfile(name): return f'{OUT}/{name}{PSUF}.csv'
os.makedirs(OUT, exist_ok=True); os.makedirs(FIG, exist_ok=True)

def style_ax(ax, title=None, xlabel=None, ylabel=None):
    ax.grid(True, color=GRID, linewidth=0.6); ax.set_axisbelow(True)
    for s in ax.spines.values(): s.set_color(AXIS)
    ax.tick_params(colors=INK2, labelsize=9)
    if title: ax.set_title(title, color=INK, fontsize=11)
    if xlabel: ax.set_xlabel(xlabel, color=INK2, fontsize=10)
    if ylabel: ax.set_ylabel(ylabel, color=INK2, fontsize=10)

def is_pointer(p): return open(p, 'rb').read(40).startswith(b'version https://git-lfs')
def need(*paths):
    for p in paths: assert not is_pointer(p), f'{p} is a Git LFS pointer: python3 tools/lfs_pull.py {p}'

SUMMARY = []      # one row per guideline check; written to Output/validation_mot/summary_<stage>.csv
def add_summary(test, ref, item, level, criterion, result, status, explanation, data):
    SUMMARY.append({'period': PERIOD, 'test': test, 'guideline': ref, 'item': item, 'level': level, 'criterion': criterion,
                    'result': result, 'status': status, 'explanation': explanation, 'data (vintage)': data})

def scatter_origin(ax, obs, mod, title, xlabel, ylabel, logscale=False, color=BLUE):
    """Guideline format: observed on x, model on y, regression through the origin with equation and R2."""
    o = np.asarray(obs, float); m = np.asarray(mod, float); ok = np.isfinite(o) & np.isfinite(m)
    o, m = o[ok], m[ok]
    ax.scatter(o, m, s=10, color=color, alpha=0.55, linewidths=0)
    hi = max(o.max(), m.max()) * 1.05 if len(o) else 1
    ax.plot([0, hi], [0, hi], color=MUTED, lw=0.8, ls=':')
    b = vm.slope_origin(o, m); r2 = vm.r2_correl(o, m)
    ax.plot([0, hi], [0, b * hi], color=ORANGE, lw=1.5)
    ax.text(0.04, 0.95, f'y = {b:.2f}x   R² = {r2:.2f}   n = {len(o)}', transform=ax.transAxes, va='top', fontsize=9, color=INK)
    if logscale: ax.set_xscale('symlog', linthresh=10); ax.set_yscale('symlog', linthresh=10)
    style_ax(ax, title, xlabel, ylabel)

def save_fig(fig, name):
    fig.savefig(f'{FIG}/{name.replace(".png", PSUF + ".png")}', dpi=150, bbox_inches='tight', facecolor='white'); plt.close(fig)

# Ministry of Transport validation, wave 3: link volumes against counts (T11 = handover step 42)

**Question.** Does the 2022 car layer, loaded on the Emme network, reproduce the counts on the 3,241 counted links (guideline 5.1: R2 above 0.85, slope 0.9-1.1, RMSE% by volume class)?
**Method.** All-or-nothing assignment of the car layer (person trips divided by the window occupancy of the survey, section 0) on free-flow shortest paths: one path per origin-destination pair, no congestion, no turn penalties, centroids not usable as through nodes.
The network has no speed field, so speeds by link `TYPE` are **assumed** (90 / 70 / 60 / 50 / 40 km/h for types 1-5, 30 for ramps and connectors; sensitivity: shortest distance). The layer is residents' personal car trips with both ends in the study area and without intra-TAZ trips (they load no link);
the counts are all traffic (trucks, vans, taxis, buses, visitors, trips with an external end). The raw comparison is therefore expected to miss on the slope; the comparison after one global factor judges the pattern, and is reported beside it (plan T11).
Counts: PCE of the Emme links divided by 1.10; count year from the `DATE` field (2021-2023 primary, all years as sensitivity; links of unknown year in their own row).

In [2]:
import geopandas as gpd, time
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra
def sq(p): m = pd.read_csv(p, index_col=0); m.index = m.index.astype(int); m.columns = m.columns.astype(int); return m
t0 = time.time(); CRS = 2039; PCE = 1.10
NET = 'Input/Network_with_Counts/Emme_Links_Final_Res 2026-09-23.shp'; need(NET.replace('.shp', '.dbf'), NET, 'Input/TAZ_North/TAZ_North.shp', 'Input/Submatrix_tazs.xlsx')
net = gpd.read_file(NET, encoding='utf-8').to_crs(CRS)
net['pce'] = net[[f'YARAM{h}' for h in HOURS]].sum(axis=1); net['counted'] = net['pce'] > 0
num = pd.to_numeric(net['DATE'], errors='coerce'); yr = pd.to_datetime(net['DATE'].where(num.isna()), format='%d/%m/%Y', errors='coerce').dt.year
yr.loc[num.notna()] = pd.to_datetime(num.dropna(), unit='D', origin='1899-12-30').dt.year; net['count_year'] = yr.where(yr.notna(), np.where(net['F2023COUNT'].notna(), 2023, np.nan))
auto = net['MODES'].fillna('').str.contains('a'); road = net[auto & (net.TYPE != 9)].copy(); conn = net[auto & (net.TYPE == 9)].copy()
print(f'{len(net):,} links; road {len(road):,}, connectors {len(conn):,}; counted road links {int(road.counted.sum()):,}; count year of the counted links:', road.loc[road.counted, 'count_year'].value_counts(dropna=False).sort_index().to_dict())
taz = gpd.read_file('Input/TAZ_North/TAZ_North.shp').to_crs(CRS); taz['TAZ'] = taz['TAZ_NUMBER'].astype(int); taz = taz.drop_duplicates('TAZ')
car = sq(f'Output/ths2017/three_mode_2022{SUF}/car_2022_taz.csv'); TAZ = list(car.index); assert set(TAZ) <= set(taz.TAZ)

29,710 links; road 25,340, connectors 2,101; counted road links 3,231; count year of the counted links: {2017.0: 1080, 2018.0: 210, 2019.0: 66, 2020.0: 323, 2021.0: 921, 2022.0: 154, 2023.0: 271, nan: 206}


## 1. Window occupancy (corrected hour, see wave 2 / T10) and the network graph

In [3]:
_tr = pd.read_excel('Input/THS_2017-2018/trips_ths_2017.xlsx').sort_values(['PerID3', 'SurveyDay', 'placeno']); _g = _tr.groupby(['PerID3', 'SurveyDay'])
_tr['trip_dep_h'] = _g['Dep_h'].shift(1); _tr['_o'] = _g['actTaz'].shift(1); _w = _tr.dropna(subset=['trip_dep_h', '_o']); _w = _w[_w.trip_dep_h.isin(HOURS)]
OCC = float(_w[_w.mainmode.isin([10, 11])]['new_wf'].sum() / _w[_w.mainmode == 10]['new_wf'].sum())
PHF = float(pd.read_csv('Output/ths2017/three_mode_2022/peak_hour_factors.csv').query("level == 'study area, all trips' and layer == 'CAR'")['PHF3h'].iloc[0]) if PERIOD == 'AM' else np.nan
VEH = car.values / OCC; print(f'{PLABEL}: car layer {car.values.sum():,.0f} person trips / occupancy {OCC:.3f} = {VEH.sum():,.0f} vehicles; intra-TAZ {np.trace(VEH):,.0f} load no link; peak-hour factor {PHF}')
SPEED = {1: 90, 2: 70, 3: 60, 4: 50, 5: 40, 8: 30, 9: 30}
def build(cost):
    """Graph: network nodes, then one origin-side and one destination-side node per centroid, then one virtual origin / destination per TAZ."""
    nodes = pd.unique(np.r_[road.INODE, road.JNODE]); ix = {n: i for i, n in enumerate(nodes)}; M0 = len(nodes)
    cen = sorted((set(conn.INODE) | set(conn.JNODE)) - set(nodes)); ci = {c: i for i, c in enumerate(cen)}; nc = len(cen)
    # location and TAZ of each centroid from its connector geometry
    loc = {}
    for r in conn.itertuples():
        g = r.geometry
        if r.INODE in ci: loc[r.INODE] = g.coords[0]
        if r.JNODE in ci: loc[r.JNODE] = g.coords[-1]
    cp = gpd.GeoDataFrame({'c': list(loc)}, geometry=gpd.points_from_xy([v[0] for v in loc.values()], [v[1] for v in loc.values()]), crs=CRS)
    cj = gpd.sjoin(cp, taz[['TAZ', 'geometry']], how='left', predicate='within').drop_duplicates('c').set_index('c')['TAZ']
    return nodes, ix, M0, cen, ci, nc, cj, loc
nodes, ix, M0, cen, ci, nc, cen_taz, loc = build(None)
lab = conn.dropna(subset=['TAZ']).copy(); lab['cen'] = [a if a in ci else b for a, b in zip(lab.INODE, lab.JNODE)]; lab_t = lab.drop_duplicates('cen').set_index('cen')['TAZ']
agree = (cen_taz.reindex(lab_t.index) == lab_t).mean()
print(f'centroids {nc:,}; located in a study TAZ polygon {int(cen_taz.notna().sum()):,}; labelled by the connector TAZ field {len(lab_t):,}; polygon and label agree for {agree:.1%} (the rest lie within tens of metres of the boundary)')
cen_taz = lab_t.reindex(cen_taz.index).where(lab_t.reindex(cen_taz.index).notna(), cen_taz)       # the connector TAZ label first (the network's own definition), the polygon second
have = set(cen_taz.dropna().astype(int)); missing = [t for t in TAZ if t not in have]
cpt = gpd.GeoDataFrame({'c': list(loc)}, geometry=gpd.points_from_xy([v[0] for v in loc.values()], [v[1] for v in loc.values()]), crs=CRS).set_index('c')
tc = taz.set_index('TAZ').geometry.centroid
extra = {}                                                                                           # TAZs with no centroid of their own: the nearest centroid takes their trips
for t in missing:
    dd = cpt.geometry.distance(tc[t]); extra[t] = (dd.idxmin(), dd.min())
print(f'TAZs of the car layer with a centroid in the network: {len(TAZ) - len(missing)} of {len(TAZ)}; the other {len(missing)} use the nearest centroid (median distance {np.median([v[1] for v in extra.values()]) if extra else 0:.0f} m); their share of the vehicle trips: origins {VEH[[TAZ.index(t) for t in missing]].sum() / VEH.sum():.1%}')
cen_pairs = [(c, int(t)) for c, t in cen_taz.dropna().items()] + [(c, t) for t, (c, _) in extra.items()]

midday 10:00-14:00: car layer 1,150,868 person trips / occupancy 1.330 = 865,007 vehicles; intra-TAZ 151,800 load no link; peak-hour factor nan
centroids 801; located in a study TAZ polygon 790; labelled by the connector TAZ field 791; polygon and label agree for 84.8% (the rest lie within tens of metres of the boundary)
TAZs of the car layer with a centroid in the network: 755 of 778; the other 23 use the nearest centroid (median distance 379 m); their share of the vehicle trips: origins 2.1%


In [4]:
def assign(cost_fn, label):
    OI = M0; DI = M0 + nc; VO = M0 + 2 * nc; VD = VO + len(TAZ); N = VD + len(TAZ); tix = {t: k for k, t in enumerate(TAZ)}
    u, v, w, lid = [], [], [], []
    rr = road.reset_index(drop=True); cw = np.maximum(cost_fn(rr), 1e-4)
    u += [ix[a] for a in rr.INODE]; v += [ix[b] for b in rr.JNODE]; w += list(cw); lid += list(range(len(rr)))
    for r in conn.itertuples():
        wt = max(r.LENGTH / SPEED[9] * 60 if label == 'time' else r.LENGTH, 1e-4)
        if r.INODE in ci and r.JNODE in ix: u.append(OI + ci[r.INODE]); v.append(ix[r.JNODE]); w.append(wt); lid.append(-1)      # centroid -> network
        elif r.JNODE in ci and r.INODE in ix: u.append(ix[r.INODE]); v.append(DI + ci[r.JNODE]); w.append(wt); lid.append(-1)    # network -> centroid
    for c, t in cen_pairs:
        if t in tix: u += [VO + tix[t], DI + ci[c]]; v += [OI + ci[c], VD + tix[t]]; w += [1e-6, 1e-6]; lid += [-1, -1]
    u, v, w, lid = map(np.array, (u, v, w, lid)); G = csr_matrix((w, (u, v)), shape=(N, N))
    key = u.astype(np.int64) * N + v; o = np.lexsort((w, key)); key_s, lid_s = key[o], lid[o]
    first = np.r_[True, key_s[1:] != key_s[:-1]]; key_s, lid_s = key_s[first], lid_s[first]            # cheapest of parallel links
    load = np.zeros(len(rr)); VEHm = VEH
    for k in range(0, len(TAZ), 80):
        idx = list(range(k, min(k + 80, len(TAZ))))
        dist, pred = dijkstra(G, directed=True, indices=[VO + i for i in idx], return_predecessors=True)
        for r_, i in enumerate(idx):
            dem = VEHm[i].copy(); dem[i] = 0
            if dem.sum() <= 0: continue
            flow = np.zeros(N); flow[VD:VD + len(TAZ)] = dem; d = dist[r_]; p = pred[r_]
            order = np.argsort(-np.where(np.isfinite(d), d, -1)); order = order[(flow[order] > 0) | True]
            fl = flow
            for n in order:
                f = fl[n]
                if f <= 0: continue
                pn = p[n]
                if pn < 0: continue
                fl[pn] += f
                j = np.searchsorted(key_s, pn * np.int64(N) + n)
                if j < len(key_s) and key_s[j] == pn * np.int64(N) + n and lid_s[j] >= 0: load[lid_s[j]] += f
    out = rr[['ID', 'INODE', 'JNODE', 'TYPE', 'LANES', 'LENGTH', 'pce', 'counted', 'count_year']].copy(); out['assigned_veh'] = load; return out
asg = {'free-flow time': assign(lambda d: d.LENGTH / d.TYPE.map(SPEED).fillna(40) * 60, 'time')}
print(f'free-flow assignment done [{time.time() - t0:.0f} s]; assigned vehicle-km {(asg["free-flow time"].assigned_veh * asg["free-flow time"].LENGTH).sum():,.0f}')
asg['shortest distance'] = assign(lambda d: d.LENGTH, 'dist'); print(f'distance assignment done [{time.time() - t0:.0f} s]')
for k, a in asg.items(): a.to_csv(vfile('T11_assignment_links_' + k.split()[0].replace('free-flow', 'time')), index=False, float_format='%.3f')

free-flow assignment done [33 s]; assigned vehicle-km 6,573,658


distance assignment done [36 s]


## 2. T11 - the comparison with the counts (guideline 5.1)

In [5]:
corr_taz = set(pd.read_excel('Input/Submatrix_tazs.xlsx').query('IsLRT_Corridor == 1')['TAZ']); zone_geom = taz[taz.TAZ.isin(corr_taz)].geometry.union_all().buffer(1000)
def prep(a):
    a = a.copy(); a['count_veh'] = a['pce'] / PCE; a['sel'] = a.counted
    mid = gpd.GeoSeries(road.reset_index(drop=True).geometry.interpolate(0.5, normalized=True), crs=CRS); a['corridor'] = mid.within(zone_geom).values
    a['yr_group'] = np.where(a.count_year.between(2021, 2023), '2021-2023', np.where(a.count_year.isna(), 'unknown year', 'before 2021')); return a
def stats(a, label, scale=1.0):
    o, m = a['count_veh'].values, a['assigned_veh'].values * scale
    s = vm.fit_stats(o, m); cl = vm.rmse_by_class(o, m, [(lo, hi, lim) for lo, hi, lim in vm.ROAD_RMSE_CLASSES]) if PERIOD == 'AM' else []
    return {'set': label, 'n links': s['n'], 'counted vehicles': s['obs_total'], 'assigned vehicles': s['mod_total'], 'total diff %': s['total_diff_pct'], 'R2': s['R2'], 'slope through origin': s['slope_origin'], 'RMSE %': s['RMSE_pct']}
res = []
for vname, a0 in asg.items():
    a = prep(a0); b = a[a.sel]
    for lab_, sub_ in [('all counted links', b), ('counts 2021-2023', b[b.yr_group == '2021-2023']), ('counts before 2021', b[b.yr_group == 'before 2021']), ('count year unknown', b[b.yr_group == 'unknown year']),
                       ('corridor sub-network', b[b.corridor]), ('rest of the network', b[~b.corridor])] + [(f'road type {t}', b[b.TYPE == t]) for t in sorted(b.TYPE.unique())]:
        if len(sub_) >= 10:
            r = stats(sub_, lab_); r['assignment'] = vname; k_ = sub_.count_veh.sum() / sub_.assigned_veh.sum() if sub_.assigned_veh.sum() > 0 else np.nan
            r2 = stats(sub_, lab_, k_); r.update({'scale factor to counts': k_, 'R2 after scaling': r2['R2'], 'slope after scaling': r2['slope through origin'], 'RMSE % after scaling': r2['RMSE %']}); res.append(r)
t11 = pd.DataFrame(res)[['assignment', 'set', 'n links', 'counted vehicles', 'assigned vehicles', 'total diff %', 'R2', 'slope through origin', 'RMSE %', 'scale factor to counts', 'R2 after scaling', 'slope after scaling', 'RMSE % after scaling']]
t11.to_csv(vfile('T11_link_fit'), index=False, float_format='%.4f'); print(t11.round(3).to_string(index=False))
A = prep(asg['free-flow time']); B = A[A.sel]; PRIM = B[B.yr_group == '2021-2023']
if PERIOD == 'AM':
    pk = PRIM.copy(); pk['count_peak'] = net.set_index('ID').loc[pk.ID, 'YARAM7'].values / PCE; pk['assigned_peak'] = pk['assigned_veh'] * PHF
    cls = pd.DataFrame(vm.rmse_by_class(pk['count_peak'], pk['assigned_peak'], vm.ROAD_RMSE_CLASSES)); k_pk = pk.count_peak.sum() / pk.assigned_peak.sum()
    cls_s = pd.DataFrame(vm.rmse_by_class(pk['count_peak'], pk['assigned_peak'] * k_pk, vm.ROAD_RMSE_CLASSES)); cls = cls.merge(cls_s[['class', 'RMSE_pct', 'pass']], on='class', suffixes=('', ' after scaling')); cls.to_csv(vfile('T11_peak_hour_classes'), index=False, float_format='%.2f')
    fp = vm.fit_stats(pk.count_peak, pk.assigned_peak); fps = vm.fit_stats(pk.count_peak, pk.assigned_peak * k_pk); print(f'peak hour 07-08, counts 2021-2023, {len(pk)} links: R2 {fp["R2"]:.3f}, slope {fp["slope_origin"]:.2f}, RMSE% {fp["RMSE_pct"]:.0f}; after one factor ({k_pk:.2f}) slope {fps["slope_origin"]:.2f}, RMSE% {fps["RMSE_pct"]:.0f}'); print(cls.round(1).to_string(index=False))
# screenlines of step 36 recomputed from the assignment
cx = pd.read_csv('Output/validation/car_cordon_crossing_links.csv'); cx = cx.merge(A[['ID', 'assigned_veh']], on='ID', how='left')
sl = cx.groupby(['cordon', 'direction']).apply(lambda g: pd.Series({'links': len(g), 'counted share of PCE': g.loc[g.counted, 'pce_6_9'].sum() / max(g[[f'YARAM{h}_imp' for h in (6, 7, 8)]].sum().sum(), 1), 'count vehicles (imputed)': g[[f'YARAM{h}_imp' for h in (6, 7, 8)]].sum().sum() / PCE, 'assigned vehicles': g.assigned_veh.sum()}), include_groups=False).reset_index() if PERIOD == 'AM' else pd.DataFrame()
if len(sl): sl['assigned / count'] = sl['assigned vehicles'] / sl['count vehicles (imputed)']; sl.to_csv(vfile('T11_screenlines_from_assignment'), index=False, float_format='%.3f'); print(sl.round(2).to_string(index=False))

       assignment                  set  n links  counted vehicles  assigned vehicles  total diff %    R2  slope through origin  RMSE %  scale factor to counts  R2 after scaling  slope after scaling  RMSE % after scaling
   free-flow time    all counted links     3231       5915617.273        4121888.813       -30.322 0.477                 0.661  79.793                   1.435             0.477                0.948                92.320
   free-flow time     counts 2021-2023     1346       1953424.545        1354053.746       -30.683 0.504                 0.701  87.117                   1.443             0.504                1.012               108.677
   free-flow time   counts before 2021     1679       3400323.636        2453006.609       -27.860 0.485                 0.672  72.705                   1.386             0.485                0.932                80.514
   free-flow time   count year unknown      206        561869.091         314828.458       -43.968 0.296                

In [6]:
def row(vname, lab_):
    return t11[(t11.assignment == vname) & (t11.set == lab_)].iloc[0]
p = row('free-flow time', 'counts 2021-2023'); a_ = row('free-flow time', 'all counted links'); d_ = row('shortest distance', 'counts 2021-2023')
add_summary('T11', '5.1', f'link volumes vs counts, all-or-nothing assignment, counts 2021-2023, {PLABEL} (raw)', f'{int(p["n links"])} counted links', 'R2 above 0.85, slope 0.9-1.1, RMSE% by volume class',
            f'assigned / counted {1 + p["total diff %"] / 100:.2f}; R2 {p["R2"]:.2f}, slope {p["slope through origin"]:.2f}, RMSE% {p["RMSE %"]:.0f}; all years: R2 {a_["R2"]:.2f}, slope {a_["slope through origin"]:.2f}; shortest-distance routing: R2 {d_["R2"]:.2f}, slope {d_["slope through origin"]:.2f}',
            vm.verdict(bool(p['R2'] > 0.85 and 0.9 <= p['slope through origin'] <= 1.1 and p['RMSE %'] < 35), explained=True),
            'Residents-only layer (no trucks, vans, taxis, buses, visitors, external ends), free-flow all-or-nothing routing, assumed speeds by link type, intra-TAZ trips load no link; the slope is the share of the road the layer represents.', 'car layer 2022; Emme counts')
add_summary('T11', '5.1', f'link volumes vs counts after one global factor ({p["scale factor to counts"]:.2f}), counts 2021-2023, {PLABEL}', f'{int(p["n links"])} counted links', 'R2 above 0.85, RMSE% by volume class (pattern)',
            f'R2 {p["R2 after scaling"]:.2f}, slope {p["slope after scaling"]:.2f}, RMSE% {p["RMSE % after scaling"]:.0f}', vm.verdict(bool(p['R2 after scaling'] > 0.85 and p['RMSE % after scaling'] < 35), explained=True),
            'All-or-nothing assignment without congestion, with no route choice among alternatives and no externals concentrates volume on the shortest arterials; counts include flows the layer lacks, and these are not uniform over the network.', 'car layer 2022; Emme counts')
for lab_ in ['corridor sub-network', 'rest of the network']:
    r_ = row('free-flow time', lab_)
    add_summary('T11', '5.1', f'link volumes vs counts, {lab_}, {PLABEL}', f'{int(r_["n links"])} links', 'slope 0.9-1.1; pattern after scaling', f'assigned / counted {1 + r_["total diff %"] / 100:.2f}; R2 {r_["R2"]:.2f} (after scaling {r_["R2 after scaling"]:.2f}, RMSE% {r_["RMSE % after scaling"]:.0f})',
                vm.verdict(bool(0.9 <= r_['slope through origin'] <= 1.1 and r_['R2'] > 0.85), explained=True), 'As above; the corridor sub-network is the links within 1 km of the LRT-corridor TAZs.', 'car layer 2022; Emme counts')
if PERIOD == 'AM':
    add_summary('T11', '5.1', 'peak hour 07-08 by volume class (RMSE% against the guideline limits), counts 2021-2023', 'volume classes', 'RMSE% below the class limit', '; '.join(f'{c["class"]}: {c["RMSE_pct"]:.0f} % (limit {c["limit_pct"]:.0f}, n {c["n"]}; after one factor {c["RMSE_pct after scaling"]:.0f} %)' for _, c in cls.iterrows()),
                vm.verdict(bool(cls['pass'].all()), explained=True), 'The classes are for hourly volumes; the layer volume is the three-hour total times the study-area car peak-hour factor of step 20.', 'car layer 2022; Emme counts')
    add_summary('T11', '5.1', 'screenlines recomputed from the assignment (the six cordons of step 36)', f'{len(sl)} cordon-directions', 'within +/-15 %', f'assigned / counted: median {sl["assigned / count"].median():.2f}, range {sl["assigned / count"].min():.2f}-{sl["assigned / count"].max():.2f}; step 36 (desire-line allocation, occupancy 1.52): AM total 0.62',
                vm.verdict(bool(((sl['assigned / count'] - 1).abs() <= 0.15).all()), explained=True), 'The assignment routes through-trips and local trips on the network; the cordon crossings of step 36 were derived from the zone-to-zone volumes directly.', 'car layer 2022; Emme counts')
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), facecolor='white')
    for ax, (nm, sc) in zip(axes, [('raw', 1.0), ('after one global factor', PRIM.count_veh.sum() / PRIM.assigned_veh.sum())]):
        scatter_origin(ax, PRIM.count_veh, PRIM.assigned_veh * sc, f'Link volumes 06-09, {nm}', 'counted vehicles', 'assigned vehicles', logscale=True, color=BLUE)
    save_fig(fig, 'mot_T11_link_volumes.png')
else:
    fig, ax = plt.subplots(figsize=(6.2, 4.8), facecolor='white'); scatter_origin(ax, PRIM.count_veh, PRIM.assigned_veh, f'Link volumes {PLABEL}, raw', 'counted vehicles', 'assigned vehicles', logscale=True, color=BLUE); save_fig(fig, 'mot_T11_link_volumes.png')

## Summary of this notebook against the guideline

In [7]:
SUM = pd.DataFrame(SUMMARY); SUM.to_csv(f'{OUT}/summary_stage4{PSUF}.csv', index=False)
SUM[['test', 'guideline', 'item', 'level', 'criterion', 'result', 'status']]

,test,guideline,item,level,criterion,result,status
0,T11,5.1,"link volumes vs counts, all-or-nothing assignm...",1346 counted links,"R2 above 0.85, slope 0.9-1.1, RMSE% by volume ...","assigned / counted 0.69; R2 0.50, slope 0.70, ...",miss (explained)
1,T11,5.1,link volumes vs counts after one global factor...,1346 counted links,"R2 above 0.85, RMSE% by volume class (pattern)","R2 0.50, slope 1.01, RMSE% 109",miss (explained)
2,T11,5.1,"link volumes vs counts, corridor sub-network, ...",773 links,slope 0.9-1.1; pattern after scaling,assigned / counted 0.81; R2 0.51 (after scalin...,miss (explained)
3,T11,5.1,"link volumes vs counts, rest of the network, m...",2458 links,slope 0.9-1.1; pattern after scaling,assigned / counted 0.65; R2 0.47 (after scalin...,miss (explained)
